# Fundamentos de Aprendizaje Supervisado
### Estadística Computacional para la Toma de Decisiones

**Magíster en Ciencia de Datos e Inteligencia Artificial**
Universidad Andrés Bello

*Maidana, J.P. (2026). Apunte — versión Jupyter Notebook.*

---

Este notebook es la versión **ejecutable** del apunte. Cada concepto teórico va acompañado de código que puedes correr y modificar: los diagramas conceptuales están construidos con `matplotlib`, las demostraciones de *overfitting* / *underfitting* ajustan modelos reales, y el flujo de trabajo completo se aplica sobre un conjunto de datos sintético de *churn*.

> 💡 Busca los comentarios **`👉 EXPERIMENTA`**: marcan parámetros que vale la pena cambiar para ver cómo reacciona el modelo.

**Requisitos:** sólo `numpy`, `pandas`, `matplotlib`, `scikit-learn` y `statsmodels` — todo viene preinstalado en Google Colab, así que corre sin instalar nada.

## Configuración del entorno

Importamos librerías, fijamos una semilla para reproducibilidad y definimos un estilo de gráficos consistente. **Ejecuta esta celda primero.**

In [ ]:
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch

# Reproducibilidad
RNG = np.random.default_rng(42)
np.random.seed(42)

# Estilo de gráficos
plt.rcParams.update({
    "figure.figsize": (8, 5),
    "figure.dpi": 110,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 11,
})

# Paleta coherente con el apunte
AZUL    = "#2c7fb8"   # datos / variables independientes
VERDE   = "#41ab5d"   # modelo / balance ideal
NARANJA = "#fe9929"   # predicción / variable dependiente
ROJO    = "#e34a33"   # error / modelos malos

print("Entorno listo ✔")

### Utilidades para los diagramas conceptuales

Dos funciones auxiliares (`caja` y `flecha`) que usaremos para reproducir, en `matplotlib`, los diagramas de bloques del apunte. No es necesario entenderlas en detalle; son sólo herramientas de dibujo.

In [ ]:
def caja(ax, x, y, w, h, texto, color, fontsize=10):
    # Dibuja una caja redondeada con texto centrado.
    box = FancyBboxPatch((x, y), w, h,
                         boxstyle="round,pad=0.02,rounding_size=0.12",
                         linewidth=1.6, edgecolor="#37474f",
                         facecolor=color, alpha=0.92)
    ax.add_patch(box)
    ax.text(x + w/2, y + h/2, texto, ha="center", va="center",
            fontsize=fontsize, zorder=5)

def flecha(ax, p1, p2, texto="", dy=0.12):
    # Dibuja una flecha de p1 a p2 con etiqueta opcional encima.
    ax.annotate("", xy=p2, xytext=p1,
                arrowprops=dict(arrowstyle="-|>", lw=2.2, color="#37474f"))
    if texto:
        mx, my = (p1[0]+p2[0])/2, (p1[1]+p2[1])/2
        ax.text(mx, my + dy, texto, ha="center", va="bottom", fontsize=9.5,
                style="italic")

---
# 1. Introducción

El **aprendizaje supervisado** es uno de los pilares fundamentales del análisis estadístico y el machine learning moderno. Se utiliza cuando queremos predecir o clasificar un resultado basándonos en datos históricos donde ya conocemos ese resultado.

> ### 📦 Definición — Aprendizaje Supervisado
> Es un enfoque de modelamiento estadístico donde:
> 1. Tenemos **datos históricos** con ejemplos previos.
> 2. Para cada ejemplo, conocemos tanto las **características** ($X$) como el **resultado** ($y$).
> 3. Queremos construir un **modelo** que aprenda la relación entre $X$ e $y$.
> 4. Usamos ese modelo para **predecir** resultados en nuevos casos.
>
> Es *"supervisado"* porque tenemos un *"supervisor"* (los datos históricos con respuestas conocidas) que guía el aprendizaje del modelo.

## Analogía

Imagina que quieres aprender a identificar si un email es spam o no. En aprendizaje supervisado:

- Te dan 10,000 emails ya etiquetados como *"spam"* o *"no spam"* (**datos de entrenamiento**).
- Analizas patrones: emails spam tienen ciertas palabras, remitentes sospechosos, etc.
- Construyes reglas o un modelo basado en esos patrones.
- Cuando llega un email nuevo (sin etiqueta), tu modelo predice si es spam.

El *"supervisor"* son esas 10,000 etiquetas que te dicen la respuesta correcta.

## Componentes fundamentales

El siguiente diagrama resume el ciclo completo: aprendemos de datos etiquetados y luego predecimos sobre datos nuevos.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 3.2))
ax.set_xlim(0, 16); ax.set_ylim(0, 4); ax.axis("off")

caja(ax, 0.3, 1, 4, 2,
     "Datos Históricos\n\nCaracterísticas (X)\n+\nResultado conocido (y)", AZUL)
caja(ax, 6, 1, 4, 2,
     "Modelo\n\nAprende la relación\n$y = f(X)$", VERDE)
caja(ax, 11.7, 1, 4, 2,
     "Nuevos Datos\n\nSólo X  →  el modelo\npredice y", NARANJA)

flecha(ax, (4.4, 2), (5.95, 2), "Entrenar")
flecha(ax, (10.05, 2), (11.65, 2), "Predecir")

ax.set_title("Ciclo del aprendizaje supervisado", fontsize=13, weight="bold")
plt.tight_layout(); plt.show()

> ### 📝 Lo que aprenderás en este apunte
> 1. Diferencia entre predicción (regresión) y clasificación.
> 2. Qué son variables dependientes e independientes.
> 3. Problemas de *overfitting* y *underfitting*.
> 4. Flujo de trabajo completo en ML supervisado.
> 5. Buenas prácticas y errores comunes a evitar.

---
# 2. Predicción vs Clasificación

El tipo de variable dependiente determina si enfrentamos un problema de **regresión** (predicción numérica) o **clasificación** (asignación de categorías).

## 2.1 Problemas de Regresión (Predicción)

> ### 📦 Definición — Regresión
> **Variable dependiente:** Continua (puede tomar cualquier valor numérico).
> **Objetivo:** Predecir un **valor numérico**.
>
> **Ejemplos:**
> - Predecir precio de una casa: \$150,000, \$200,500, \$325,000
> - Predecir ventas mensuales: \$45,320, \$52,180, \$38,900
> - Predecir temperatura: 15.3 °C, 22.7 °C, 18.1 °C
> - Predecir tiempo de entrega: 2.5 días, 3.8 días, 1.2 días
>
> **Modelos típicos:** regresión lineal simple, regresión lineal múltiple, regresión polinomial.
>
> **Métricas de evaluación:** RMSE, MAE, R², MAPE.

La siguiente celda **ajusta de verdad** una recta a datos con tendencia lineal y ruido.

In [ ]:
from sklearn.linear_model import LinearRegression

# Datos sintéticos con tendencia lineal + ruido
n = 40
X = np.linspace(0, 10, n).reshape(-1, 1)
y = 0.55 * X.ravel() + 1.0 + RNG.normal(0, 0.45, n)   # 👉 EXPERIMENTA: cambia el 0.45 (ruido)

# Ajuste de una regresión lineal
reg = LinearRegression().fit(X, y)
x_grid = np.linspace(0, 10, 100).reshape(-1, 1)
y_hat = reg.predict(x_grid)

fig, ax = plt.subplots()
ax.scatter(X, y, color=AZUL, s=35, label="Datos observados")
ax.plot(x_grid, y_hat, color=ROJO, lw=2.5,
        label=fr"$\hat{{y}} = {reg.coef_[0]:.2f}\,X + {reg.intercept_:.2f}$")
ax.set_xlabel("X"); ax.set_ylabel("y")
ax.set_title("Regresión: predecimos un valor continuo", weight="bold")
ax.legend()
plt.tight_layout(); plt.show()

## 2.2 Problemas de Clasificación

> ### 📦 Definición — Clasificación
> **Variable dependiente:** Categórica (pertenece a una de varias clases).
> **Objetivo:** Asignar una **clase** o **categoría**.
>
> **Clasificación Binaria** (dos clases):
> - ¿El cliente comprará? → Sí / No
> - ¿El email es spam? → Spam / No spam
> - ¿El paciente tiene la enfermedad? → Positivo / Negativo
> - ¿El cliente abandonará? (*churn*) → Churn / No churn
>
> **Clasificación Multiclase** (más de dos clases):
> - Tipo de flor: Setosa / Versicolor / Virginica
> - Nivel de satisfacción: Muy insatisfecho … Muy satisfecho
> - Categoría de producto: Electrónica / Ropa / Alimentos / Hogar / Otros
>
> **Modelos típicos:** regresión logística, árboles de decisión, SVM.
>
> **Métricas de evaluación:** Accuracy, Precision, Recall, F1-Score, ROC-AUC.

A diferencia de la regresión, aquí el modelo aprende una **frontera de decisión** que separa las clases. La celda ajusta una regresión logística y dibuja esa frontera.

In [ ]:
from sklearn.linear_model import LogisticRegression

# Dos nubes de puntos (dos clases) en 2D
n_c = 60
clase0 = RNG.normal(loc=[2.0, 2.0], scale=0.7, size=(n_c, 2))
clase1 = RNG.normal(loc=[5.0, 5.0], scale=0.7, size=(n_c, 2))   # 👉 EXPERIMENTA: acerca las nubes (5.0 -> 3.5)
Xc = np.vstack([clase0, clase1])
yc = np.array([0]*n_c + [1]*n_c)

clf = LogisticRegression().fit(Xc, yc)

# Malla para pintar la frontera de decisión
xx, yy = np.meshgrid(np.linspace(0, 7, 300), np.linspace(0, 7, 300))
Z = clf.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

fig, ax = plt.subplots()
ax.contourf(xx, yy, Z, alpha=0.15, colors=[AZUL, ROJO], levels=[-0.5, 0.5, 1.5])
ax.scatter(*clase0.T, color=AZUL, s=30, label="Clase 0")
ax.scatter(*clase1.T, color=ROJO, s=30, label="Clase 1")
ax.contour(xx, yy, Z, levels=[0.5], colors=[VERDE], linewidths=2.5, linestyles="--")
ax.set_xlabel(r"$X_1$"); ax.set_ylabel(r"$X_2$")
ax.set_title("Clasificación: el modelo aprende una frontera de decisión", weight="bold")
ax.legend(loc="upper left")
plt.tight_layout(); plt.show()

## 2.3 Comparación

| Aspecto | Regresión | Clasificación |
|---|---|---|
| Variable objetivo | Continua (numérica) | Categórica (discreta) |
| Salida del modelo | Valores numéricos | Clases o probabilidades |
| Ejemplo | Predecir precio: \$150.5 | Predecir spam: Sí/No |
| Visualización | Línea/curva de ajuste | Frontera de decisión |
| Métrica principal | RMSE, R² | Accuracy, Recall |
| Interpretación del error | Distancia al valor real | Proporción mal clasificada |

## 2.4 ¿Cómo decidir?

> ### ❗ Regla simple
> **Pregúntate:** ¿la respuesta es un número o una categoría?
> - **Número continuo** → Regresión
> - **Categoría o clase** → Clasificación

> ### 🧪 Ejemplos de decisión
> - **¿Cuánto venderemos el próximo mes?** → Regresión (numérico: \$45,000)
> - **¿Este cliente comprará o no?** → Clasificación (Sí/No)
> - **¿Qué calificación tendrá un estudiante?** → Regresión (nota 0–100, continua)
> - **¿El estudiante aprobará o reprobará?** → Clasificación (Aprobado/Reprobado)
> - **¿Cuál será la temperatura mañana?** → Regresión (23.5 °C, continua)
> - **¿Lloverá mañana?** → Clasificación (Sí/No)

---
# 3. Variables Dependientes e Independientes

## 3.1 Variable Dependiente ($y$)

> ### 📦 Definición — Variable Dependiente
> Es la variable que queremos **predecir** o **explicar**. Depende de otras variables.
>
> **Características:**
> - Es el **objetivo** del modelo.
> - Su valor es conocido en datos históricos.
> - Su valor es desconocido en datos nuevos (lo que queremos predecir).
> - Determina el tipo de problema (regresión o clasificación).
>
> **Notación común:** $y$, $Y$, *target*, *label*, respuesta.

| Industria | Variable Dependiente |
|---|---|
| E-commerce | Monto de compra, probabilidad de conversión |
| Banca | Probabilidad de default, riesgo crediticio |
| Marketing | Tasa de conversión, ROI de campaña |
| Salud | Diagnóstico de enfermedad, tiempo de recuperación |
| Retail | Demanda del producto, rotación de inventario |
| Recursos Humanos | Rotación de empleados, satisfacción laboral |

## 3.2 Variables Independientes ($X$)

> ### 📦 Definición — Variables Independientes
> Son las variables que usamos para **predecir** o **explicar** la variable dependiente.
>
> **Características:**
> - Son los **predictores** o *features*.
> - Su valor es conocido tanto en datos históricos como nuevos.
> - Pueden ser múltiples (regresión múltiple).
> - Pueden ser de diferentes tipos (numéricas, categóricas).
>
> **Notación común:** $X$, $X_1, X_2, X_3, \ldots$, *features*, predictores, covariables.

### Tipos de variables independientes

| Tipo | Descripción | Ejemplos |
|---|---|---|
| **Continuas** | Cualquier valor en un rango | edad, ingreso, temperatura |
| **Discretas** | Valores enteros | n° hijos, visitas, años experiencia |
| **Categóricas Nominales** | Categorías sin orden | género, color, país |
| **Categóricas Ordinales** | Categorías con orden | educación, satisfacción |

## 3.3 Relación entre variables

Conceptualmente, el modelo combina todas las variables independientes para producir la variable dependiente:

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4.5))
ax.set_xlim(0, 12); ax.set_ylim(-2.5, 4); ax.axis("off")

# Variables independientes (círculos)
ys = [3.2, 2.0, 0.8, -1.4]
labels = ["$X_1$", "$X_2$", "$X_3$", "$X_n$"]
for yy, lab in zip(ys, labels):
    c = plt.Circle((1.2, yy), 0.42, facecolor=AZUL, edgecolor="#37474f", lw=1.6, alpha=0.92)
    ax.add_patch(c)
    ax.text(1.2, yy, lab, ha="center", va="center", fontsize=12, zorder=5)
ax.text(1.2, -0.3, r"$\vdots$", ha="center", va="center", fontsize=18)
ax.text(1.2, 3.95, "Independientes", ha="center", fontsize=10, style="italic")

# Modelo
caja(ax, 5, 0.1, 2.4, 1.8, "Modelo\n$f(X)$", VERDE, fontsize=12)

# Variable dependiente
c = plt.Circle((10, 1), 0.6, facecolor=NARANJA, edgecolor="#37474f", lw=1.6, alpha=0.92)
ax.add_patch(c)
ax.text(10, 1, "$y$", ha="center", va="center", fontsize=14, zorder=5)
ax.text(10, 1.95, "Dependiente", ha="center", fontsize=10, style="italic")

for yy in ys:
    ax.annotate("", xy=(4.95, 1), xytext=(1.65, yy),
                arrowprops=dict(arrowstyle="-|>", lw=1.6, color="#90a4ae"))
flecha(ax, (7.45, 1), (9.35, 1))

ax.set_title("Las variables independientes alimentan al modelo, que produce la dependiente",
             fontsize=12, weight="bold")
plt.tight_layout(); plt.show()

**Ecuación general:**

$$ y = f(X_1, X_2, X_3, \ldots, X_n) + \epsilon $$

donde:
- $y$ = variable dependiente
- $X_1, X_2, \ldots, X_n$ = variables independientes
- $f(\cdot)$ = función que relaciona $X$ con $y$ (el modelo)
- $\epsilon$ = error aleatorio (lo que el modelo no puede explicar)

> ### ❗ Interpretación
> - El modelo intenta encontrar la función $f(\cdot)$ que mejor relaciona $X$ con $y$.
> - No hay relación perfecta (siempre hay error $\epsilon$).
> - Nuestro objetivo es **minimizar** ese error.
> - Un buen modelo captura la mayor parte de la relación, dejando sólo ruido aleatorio en $\epsilon$.

## 3.4 Ejemplo completo: predicción de *churn*

**Contexto:** predecir si un cliente abandonará el servicio.

**Variable dependiente ($y$):** `churn` — 0 = No abandona, 1 = Abandona. Tipo: binaria (clasificación).

**Variables independientes ($X$):**

| Variable | Tipo | Descripción |
|---|---|---|
| `meses_cliente` | numérica continua | Antigüedad en meses |
| `llamadas_soporte` | numérica discreta | N° de llamadas a soporte |
| `gasto_mensual` | numérica continua | Gasto promedio en \$ |
| `tiene_contrato` | categórica binaria | 0 = No, 1 = Sí |
| `nivel_satisfaccion` | categórica ordinal | 1 = Bajo, 2 = Medio, 3 = Alto |
| `plan` | categórica nominal | basico / estandar / premium |

El apunte muestra esta estructura (4 filas de ejemplo):

| meses | llamadas | gasto | contrato | satisf. | **churn** |
|---|---|---|---|---|---|
| 24 | 2 | 45.50 | 1 | 3 | 0 |
| 6 | 5 | 30.00 | 0 | 1 | 1 |
| 36 | 1 | 60.00 | 1 | 3 | 0 |
| 12 | 8 | 25.00 | 0 | 1 | 1 |

A continuación generamos una versión **sintética y más grande** (1,000 clientes) de ese mismo conjunto. Lo usaremos en la Sección 5 para recorrer el flujo de trabajo completo.

In [ ]:
def generar_datos_churn(n=1000, seed=42):
    # Genera un dataset sintético de churn coherente con el ejemplo del apunte.
    rng = np.random.default_rng(seed)
    meses     = rng.integers(1, 60, n)
    llamadas  = rng.poisson(3, n)
    gasto     = np.clip(rng.normal(45, 15, n), 5, None).round(2)
    contrato  = rng.binomial(1, 0.55, n)
    satisf    = rng.choice([1, 2, 3], size=n, p=[0.3, 0.4, 0.3])
    plan      = rng.choice(["basico", "estandar", "premium"], size=n, p=[0.4, 0.4, 0.2])

    # Probabilidad de churn: baja antigüedad, muchas llamadas, baja satisfacción
    # y sin contrato aumentan el riesgo de abandono.
    logit = (1.5
             - 0.05 * meses
             + 0.35 * llamadas
             - 0.90 * satisf
             - 1.20 * contrato
             + 0.015 * (gasto - 45))
    prob  = 1 / (1 + np.exp(-logit))
    churn = rng.binomial(1, prob)

    return pd.DataFrame({
        "meses_cliente": meses,
        "llamadas_soporte": llamadas,
        "gasto_mensual": gasto,
        "tiene_contrato": contrato,
        "nivel_satisfaccion": satisf,
        "plan": plan,
        "churn": churn,
    })

df = generar_datos_churn()
print("Dimensiones:", df.shape)
print("Tasa de churn: {:.1%}".format(df['churn'].mean()))
df.head()

In [ ]:
# Estadísticas descriptivas rápidas
df.describe(include="all").T

---
# 4. Overfitting y Underfitting

Uno de los problemas más importantes en aprendizaje supervisado es encontrar el **balance correcto** entre un modelo demasiado simple y uno demasiado complejo.

Para verlo "en vivo", generamos un conjunto de datos con una relación **no lineal** conocida más ruido. Usaremos **los mismos datos** para ilustrar los tres regímenes (subajuste, balance, sobreajuste).

In [ ]:
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LinearRegression

# Relación verdadera (desconocida para el modelo) + ruido
def f_real(x):
    return np.sin(1.2 * x) + 0.3 * x

n = 30
x = np.sort(RNG.uniform(0, 6, n))
y = f_real(x) + RNG.normal(0, 0.35, n)   # 👉 EXPERIMENTA: cambia el ruido (0.35) o n (30)

X = x.reshape(-1, 1)
x_grid = np.linspace(0, 6, 400).reshape(-1, 1)

def ajustar_polinomio(grado, Xtr, ytr):
    modelo = make_pipeline(PolynomialFeatures(grado), LinearRegression())
    modelo.fit(Xtr, ytr)
    return modelo

print(f"Datos generados: {n} puntos de y = sin(1.2x) + 0.3x + ruido")

## 4.1 Underfitting (Subajuste)

> ### 📦 Definición — Underfitting
> El modelo es **demasiado simple** y no captura adecuadamente los patrones en los datos.
>
> **Características:**
> - Desempeño **pobre** tanto en entrenamiento como en prueba.
> - El modelo no ha aprendido suficiente.
> - **Bias alto** (sesgo), **Variance baja**.
>
> **Señales:** R² bajo en entrenamiento *y* en prueba, errores altos en ambos conjuntos, predicciones muy básicas, el modelo ignora patrones obvios.

Ajustamos una recta (grado 1) a datos curvos: no alcanza a seguir la curvatura.

In [ ]:
modelo_under = ajustar_polinomio(1, X, y)   # grado 1 -> recta

fig, ax = plt.subplots()
ax.scatter(x, y, color=AZUL, s=35, label="Datos")
ax.plot(x_grid, f_real(x_grid.ravel()), color="#9e9e9e", lw=1.5, ls=":", label="Relación real")
ax.plot(x_grid, modelo_under.predict(x_grid), color=ROJO, lw=2.5, label="Modelo (grado 1)")
ax.set_xlabel("X"); ax.set_ylabel("y")
ax.set_title("UNDERFITTING: la recta no captura la curvatura", weight="bold", color=ROJO)
ax.legend()
plt.tight_layout(); plt.show()

**Soluciones para underfitting:** usar un modelo más complejo, agregar variables relevantes, reducir la regularización, entrenar por más tiempo, conseguir más datos.

## 4.2 Overfitting (Sobreajuste)

> ### 📦 Definición — Overfitting
> El modelo es **demasiado complejo** y memoriza los datos de entrenamiento, incluyendo el ruido.
>
> **Características:**
> - Desempeño **excelente** en entrenamiento pero **pobre** en prueba.
> - El modelo memorizó en lugar de aprender patrones generales.
> - **Bias bajo** (sesgo), **Variance alta**.
>
> **Señales:** R² muy alto en entrenamiento pero bajo en prueba, gran diferencia entre error de entrenamiento y de prueba, predicciones perfectas en datos conocidos y malas en nuevos, coeficientes con magnitudes muy grandes.

Ajustamos un polinomio de grado alto: pasa muy cerca de cada punto, pero "inventa" oscilaciones que no existen en la relación real.

In [ ]:
GRADO_ALTO = 15   # 👉 EXPERIMENTA: sube o baja el grado y observa las oscilaciones
modelo_over = ajustar_polinomio(GRADO_ALTO, X, y)

fig, ax = plt.subplots()
ax.scatter(x, y, color=AZUL, s=35, label="Datos")
ax.plot(x_grid, f_real(x_grid.ravel()), color="#9e9e9e", lw=1.5, ls=":", label="Relación real")
ax.plot(x_grid, modelo_over.predict(x_grid), color=ROJO, lw=2.5,
        label=f"Modelo (grado {GRADO_ALTO})")
ax.set_ylim(y.min() - 1.5, y.max() + 1.5)
ax.set_xlabel("X"); ax.set_ylabel("y")
ax.set_title("OVERFITTING: el modelo memoriza el ruido y no generaliza",
             weight="bold", color=ROJO)
ax.legend()
plt.tight_layout(); plt.show()

**Soluciones para overfitting:** usar un modelo más simple, reducir el número de variables (selección), aplicar regularización (Ridge, Lasso), conseguir más datos, usar validación cruzada.

## 4.3 Modelo ideal (balance óptimo)

> ### 📦 Definición — Modelo Ideal
> Captura los **patrones verdaderos** sin memorizar el ruido.
>
> **Características:** buen desempeño en entrenamiento *y* en prueba, pequeña diferencia entre ambos errores, generaliza bien a datos nuevos, balance entre *bias* y *variance*.

Un polinomio de grado moderado (3) sigue la tendencia sin perseguir cada punto:

In [ ]:
modelo_bueno = ajustar_polinomio(3, X, y)   # grado moderado

fig, ax = plt.subplots()
ax.scatter(x, y, color=AZUL, s=35, label="Datos")
ax.plot(x_grid, f_real(x_grid.ravel()), color="#9e9e9e", lw=1.5, ls=":", label="Relación real")
ax.plot(x_grid, modelo_bueno.predict(x_grid), color=VERDE, lw=2.8, label="Modelo (grado 3)")
ax.set_xlabel("X"); ax.set_ylabel("y")
ax.set_title("BALANCE ÓPTIMO: captura la tendencia y generaliza", weight="bold", color=VERDE)
ax.legend()
plt.tight_layout(); plt.show()

## 4.4 Bias–Variance Tradeoff

> ### ❗ Compromiso fundamental
> Hay un compromiso inevitable entre *bias* (sesgo) y *variance* (varianza).
>
> **Bias (sesgo):** error por suposiciones incorrectas del modelo. Modelo simple → bias alto. *"¿Qué tan equivocado está el modelo en promedio?"*
>
> **Variance (varianza):** error por sensibilidad a fluctuaciones en los datos de entrenamiento. Modelo complejo → variance alta. *"¿Qué tan diferente sería el modelo con datos ligeramente distintos?"*
>
> $$ \text{Error Total} = \text{Bias}^2 + \text{Variance} + \text{Ruido irreducible} $$

La mejor forma de internalizar esto es **medirlo**. La siguiente celda ajusta polinomios de grado creciente y compara el error en **entrenamiento** vs **prueba**. El error de prueba dibuja una "U": baja mientras el modelo gana flexibilidad útil, y vuelve a subir cuando empieza a sobreajustar.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

# Generamos más datos para que el conjunto de prueba sea informativo
n_bv = 120
x_bv = np.sort(RNG.uniform(0, 6, n_bv))
y_bv = f_real(x_bv) + RNG.normal(0, 0.35, n_bv)
X_bv = x_bv.reshape(-1, 1)

X_tr, X_te, y_tr, y_te = train_test_split(X_bv, y_bv, test_size=0.35, random_state=42)

grados = range(1, 16)
rmse_train, rmse_test = [], []
for g in grados:
    m = ajustar_polinomio(g, X_tr, y_tr)
    rmse_train.append(np.sqrt(mean_squared_error(y_tr, m.predict(X_tr))))
    rmse_test.append(np.sqrt(mean_squared_error(y_te, m.predict(X_te))))

mejor_grado = grados[int(np.argmin(rmse_test))]

fig, ax = plt.subplots()
ax.plot(list(grados), rmse_train, "o-", color=AZUL, label="Error en entrenamiento")
ax.plot(list(grados), rmse_test, "s-", color=ROJO, label="Error en prueba")
ax.axvline(mejor_grado, color=VERDE, ls="--", lw=2, label=f"Mejor grado = {mejor_grado}")
ax.set_xlabel("Complejidad del modelo (grado del polinomio)")
ax.set_ylabel("RMSE")
ax.set_title("Bias–Variance: el error de prueba tiene forma de U", weight="bold")
ax.annotate("Underfitting\n(bias alto)", xy=(1, rmse_test[0]), xytext=(1.5, rmse_test[0]+0.15),
            fontsize=9, color="#555")
ax.annotate("Overfitting\n(variance alta)", xy=(15, rmse_test[-1]), xytext=(10.5, rmse_test[-1]+0.05),
            fontsize=9, color="#555")
ax.legend()
plt.tight_layout(); plt.show()

print(f"El grado con menor error de prueba es {mejor_grado}.")

## 4.5 Diagnóstico y soluciones

| Síntoma | Error Train | Error Test | Problema | Solución |
|---|---|---|---|---|
| Ambos altos | Alto | Alto | Underfitting | Modelo más complejo |
| Train bajo, Test alto | Bajo | Alto | Overfitting | Más datos / Regularización |
| Ambos bajos | Bajo | Bajo | ✅ Bueno | Mantener |
| Diferencia grande | Bajo | Muy Alto | Overfitting severo | Simplificar modelo |

---
# 5. Flujo de Trabajo en Machine Learning

Un proyecto de aprendizaje supervisado sigue un flujo estructurado. Seguir este proceso aumenta la probabilidad de éxito.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 6))
ax.set_xlim(0, 12); ax.set_ylim(0, 10); ax.axis("off")

fases = [
    "1. Definición\ndel Problema",
    "2. Recolección\nde Datos",
    "3. Exploración\ny Limpieza",
    "4. Preparación\nde Datos",
    "5. Entrenamiento\ndel Modelo",
    "6. Evaluación",
]
y0 = 9
for i, f in enumerate(fases):
    caja(ax, 1, y0 - i*1.5, 3.2, 1.1, f, AZUL, fontsize=9.5)
    if i > 0:
        flecha(ax, (2.6, y0 - (i-1)*1.5), (2.6, y0 - i*1.5 + 1.1))

# Fases 7 y 8 a la derecha
caja(ax, 7.5, y0 - 5*1.5, 3.2, 1.1, "7. Iteración\ny Mejora", VERDE, fontsize=9.5)
caja(ax, 7.5, y0 - 4*1.5, 3.2, 1.1, "8. Implementación\ny Monitoreo", NARANJA, fontsize=9.5)

flecha(ax, (4.25, y0 - 5*1.5 + 0.55), (7.45, y0 - 5*1.5 + 0.55))   # eval -> iteración
flecha(ax, (9.1, y0 - 5*1.5 + 1.1), (9.1, y0 - 4*1.5))             # iteración -> implementación

# Flecha de retorno (iteración -> entrenamiento)
ax.annotate("", xy=(4.25, y0 - 4*1.5 + 0.55), xytext=(7.45, y0 - 5*1.5 + 0.9),
            arrowprops=dict(arrowstyle="-|>", lw=1.8, color=ROJO,
                            connectionstyle="arc3,rad=-0.3", ls="--"))
ax.text(5.8, y0 - 4.3*1.5, "retorno", color=ROJO, fontsize=9, style="italic")

ax.set_title("Flujo de trabajo de un proyecto de ML supervisado", fontsize=13, weight="bold")
plt.tight_layout(); plt.show()

## Fase 1: Definición del Problema

> ### 🔧 Método
> **Objetivo:** entender claramente qué queremos lograr.
>
> **Preguntas clave:** ¿Qué predecir exactamente? ¿Regresión o clasificación? ¿Por qué importa (valor de negocio)? ¿Qué error es aceptable? ¿Cómo se usarán las predicciones?

> ### 🧪 Ejemplo
> **Pregunta vaga:** *"Queremos predecir ventas"*.
>
> **Pregunta bien definida:**
> - **Qué:** predecir ventas mensuales en dólares de cada tienda.
> - **Por qué:** optimizar inventario y reducir quiebres de stock.
> - **Tipo:** regresión (valor numérico).
> - **Error aceptable:** ±10 % del valor real.
> - **Uso:** decisiones de compra al proveedor 2 semanas antes.

## Fase 2: Recolección de Datos

> ### 🔧 Método
> **Objetivo:** obtener datos históricos suficientes y relevantes.
>
> **Consideraciones:** cantidad (más datos → mejor modelo, en general), calidad (correctos, actualizados, representativos), relevancia (variables que impactan el objetivo), período (balance entre recencia e historia).
>
> **Regla de oro:** mínimo 10–20 observaciones por variable independiente.

## Fase 3: Exploración y Limpieza

> ### 🔧 Método
> **Objetivo:** entender los datos y prepararlos para el modelamiento.
>
> **Análisis Exploratorio (EDA):** estadísticas descriptivas (media, mediana, desviación), visualizaciones (histogramas, scatter, boxplots), identificación de problemas (faltantes, outliers, errores).
>
> **Limpieza:** tratar faltantes (eliminar o imputar), manejar outliers, corregir errores de captura, eliminar duplicados.

Un EDA mínimo sobre nuestro dataset de *churn*:

In [ ]:
# EDA breve sobre el dataset de churn
fig, axes = plt.subplots(1, 3, figsize=(13, 3.6))

axes[0].hist(df["meses_cliente"], bins=20, color=AZUL, alpha=0.8)
axes[0].set_title("Antigüedad (meses)"); axes[0].set_xlabel("meses_cliente")

axes[1].hist(df["gasto_mensual"], bins=20, color=VERDE, alpha=0.8)
axes[1].set_title("Gasto mensual"); axes[1].set_xlabel("gasto_mensual")

tasa = df.groupby("nivel_satisfaccion")["churn"].mean()
axes[2].bar(tasa.index.astype(str), tasa.values, color=NARANJA, alpha=0.9)
axes[2].set_title("Tasa de churn por satisfacción")
axes[2].set_xlabel("nivel_satisfaccion"); axes[2].set_ylabel("P(churn)")

plt.tight_layout(); plt.show()

print("Valores faltantes por columna:")
print(df.isna().sum())

## Fase 4: Preparación de Datos

> ### 🔧 Método
> **Objetivo:** transformar los datos para que el modelo pueda procesarlos.
>
> Tareas clave: **división train/test** (¡crítico!), **codificación** de categóricas (one-hot para nominales, label/ordinal para ordinales), **escalamiento** (estandarización o normalización) y **creación de variables** (interacciones, transformaciones).

> ### 🚨 Regla de oro
> ## NUNCA uses datos de *test* para entrenar
> La división train/test debe hacerse **ANTES** de cualquier análisis o transformación, y los datos de test deben permanecer "ocultos" hasta la evaluación final.
>
> **¿Por qué?** Si "miras" los datos de test antes, tu modelo puede adaptarse a ellos indirectamente (*data leakage*), produciendo evaluaciones optimistas.

Apliquemos la preparación completa sobre el dataset de *churn*:

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# --- Separar X (predictores) e y (objetivo) ---
y_churn = df["churn"]
X_churn = df.drop(columns="churn")

# 1) DIVISIÓN TRAIN/TEST  (¡lo primero de todo!)
X_train, X_test, y_train, y_test = train_test_split(
    X_churn, y_churn, test_size=0.2, random_state=42, stratify=y_churn
)
print(f"Entrenamiento: {X_train.shape[0]} filas | Prueba: {X_test.shape[0]} filas")

# 2) CODIFICACIÓN DE CATEGÓRICAS
#    'plan' es nominal  -> one-hot encoding
#    'nivel_satisfaccion' ya viene como entero ordinal (1,2,3) -> se deja igual
X_train = pd.get_dummies(X_train, columns=["plan"], drop_first=True)
X_test  = pd.get_dummies(X_test,  columns=["plan"], drop_first=True)
# Alinear columnas por si alguna categoría no apareció en test
X_train, X_test = X_train.align(X_test, join="left", axis=1, fill_value=0)

# 3) ESCALAMIENTO (estandarización: media 0, desviación 1)
#    El scaler se AJUSTA sólo con train y luego se aplica a test.
scaler = StandardScaler()
X_train_esc = scaler.fit_transform(X_train)   # fit + transform en TRAIN
X_test_esc  = scaler.transform(X_test)         # sólo transform en TEST

print("Columnas finales:", list(X_train.columns))
pd.DataFrame(X_train_esc, columns=X_train.columns).head()

## Fase 5: Entrenamiento del Modelo

> ### 🔧 Método
> **Objetivo:** ajustar el modelo a los datos de entrenamiento.
>
> Para este curso: **regresión lineal múltiple** (predicción numérica) y **regresión logística** (clasificación binaria).

### 5.a Clasificación — Regresión logística sobre *churn* (con `scikit-learn`)

In [ ]:
from sklearn.linear_model import LogisticRegression

modelo_log = LogisticRegression(max_iter=1000)
modelo_log.fit(X_train_esc, y_train)   # Solo usa TRAIN

print("Modelo de churn entrenado ✔")
print("\nImportancia (coeficientes estandarizados):")
coefs = pd.Series(modelo_log.coef_[0], index=X_train.columns).sort_values()
print(coefs.round(3))

### 5.b Regresión — Predicción de precios con `scikit-learn` y `statsmodels`

Para ilustrar la regresión numérica y la interpretación estadística clásica, generamos un pequeño dataset de precios de vivienda.

In [ ]:
from sklearn.linear_model import LinearRegression

# Dataset sintético de precios de vivienda
n_h = 300
metros      = RNG.uniform(40, 200, n_h)
habitaciones = RNG.integers(1, 6, n_h)
antiguedad  = RNG.uniform(0, 40, n_h)
precio = (1000*metros + 5000*habitaciones - 800*antiguedad
          + 50000 + RNG.normal(0, 15000, n_h))

viv = pd.DataFrame({"metros": metros, "habitaciones": habitaciones,
                    "antiguedad": antiguedad, "precio": precio})

Xv = viv[["metros", "habitaciones", "antiguedad"]]
yv = viv["precio"]
Xv_tr, Xv_te, yv_tr, yv_te = train_test_split(Xv, yv, test_size=0.2, random_state=42)

# --- scikit-learn ---
reg_viv = LinearRegression().fit(Xv_tr, yv_tr)
print("Coeficientes (scikit-learn):")
for nombre, c in zip(Xv.columns, reg_viv.coef_):
    print(f"  {nombre:14s}: {c:10.2f}")
print(f"  {'intercepto':14s}: {reg_viv.intercept_:10.2f}")

In [ ]:
import statsmodels.api as sm

# --- statsmodels: mismo ajuste, con tabla estadística completa ---
Xv_tr_const = sm.add_constant(Xv_tr)
modelo_sm = sm.OLS(yv_tr, Xv_tr_const).fit()
print(modelo_sm.summary())

## Fase 6: Evaluación

> ### 🔧 Método
> **Objetivo:** medir qué tan bien funciona el modelo en datos nuevos (test).
>
> **Comparación train vs test:**
> - Ambos buenos → modelo exitoso ✅
> - Train bueno, test malo → overfitting ✖
> - Ambos malos → underfitting ✖

### 6.a Métricas de clasificación (*churn*)

In [ ]:
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, classification_report)

y_pred = modelo_log.predict(X_test_esc)

print("Accuracy : {:.3f}".format(accuracy_score(y_test, y_pred)))
print("Precision: {:.3f}".format(precision_score(y_test, y_pred)))
print("Recall   : {:.3f}".format(recall_score(y_test, y_pred)))
print("F1-Score : {:.3f}".format(f1_score(y_test, y_pred)))
print("\nReporte completo:\n")
print(classification_report(y_test, y_pred, target_names=["No churn", "Churn"]))

# Matriz de confusión
cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(4.5, 4))
im = ax.imshow(cm, cmap="Blues")
for i in range(2):
    for j in range(2):
        ax.text(j, i, cm[i, j], ha="center", va="center",
                color="white" if cm[i, j] > cm.max()/2 else "black", fontsize=14)
ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
ax.set_xticklabels(["No churn", "Churn"]); ax.set_yticklabels(["No churn", "Churn"])
ax.set_xlabel("Predicho"); ax.set_ylabel("Real")
ax.set_title("Matriz de confusión", weight="bold")
ax.grid(False)
plt.tight_layout(); plt.show()

### 6.b Métricas de regresión (precios) y comparación train vs test

In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

def metricas_reg(modelo, X, y):
    pred = modelo.predict(X)
    return {
        "RMSE": np.sqrt(mean_squared_error(y, pred)),
        "MAE":  mean_absolute_error(y, pred),
        "R2":   r2_score(y, pred),
    }

m_train = metricas_reg(reg_viv, Xv_tr, yv_tr)
m_test  = metricas_reg(reg_viv, Xv_te, yv_te)

print(f"{'Métrica':8s}{'Train':>14s}{'Test':>14s}")
for k in ["RMSE", "MAE", "R2"]:
    print(f"{k:8s}{m_train[k]:14.3f}{m_test[k]:14.3f}")

print("\nR² similar en train y test  ->  el modelo generaliza bien ✅")

## Fase 7: Iteración y Mejora

> ### 🔧 Método
> **Objetivo:** mejorar el modelo mediante ajustes.
>
> **Estrategias:** selección de variables (eliminar no significativas), transformaciones (log, interacciones, polinomios), tratamiento de outliers, conseguir más datos, regularización (Ridge/Lasso si hay overfitting).
>
> **Proceso iterativo:** Entrenar → Evaluar → Ajustar → Repetir.

## Fase 8: Implementación y Monitoreo

> ### 🔧 Método
> **Objetivo:** poner el modelo en producción y mantenerlo.
>
> **Implementación:** guardar el modelo (pickle/joblib), crear pipeline de preprocesamiento, desarrollar API o interfaz, documentar uso y limitaciones, definir plan de actualización.
>
> **Monitoreo continuo:** tracking de predicciones vs realidad, detección de *data drift*, métricas en tiempo real, alertas de degradación.
>
> **Re-entrenamiento:** actualizaciones periódicas, re-entrenar cuando el desempeño decae, incorporar datos nuevos.

A modo de cierre práctico, así se guarda y se vuelve a cargar un modelo entrenado:

In [ ]:
import joblib, io

# Guardar el modelo y el scaler juntos (en memoria, para el ejemplo)
buffer = io.BytesIO()
joblib.dump({"modelo": modelo_log, "scaler": scaler,
             "columnas": list(X_train.columns)}, buffer)
print(f"Artefacto serializado: {buffer.tell()} bytes")

# Volver a cargarlo y predecir un cliente nuevo
buffer.seek(0)
artefacto = joblib.load(buffer)
print("Modelo recargado ✔  ->  listo para producción")

## Buenas prácticas

> ### ✅ HACER
> - Dividir los datos **ANTES** de cualquier análisis.
> - Nunca tocar los datos de test hasta la evaluación final.
> - Documentar cada decisión tomada.
> - Validar los supuestos del modelo.
> - Comparar múltiples modelos.
> - Interpretar los coeficientes antes de usar el modelo.
> - Considerar las implicancias de negocio.

> ### ✖ NO HACER
> - Entrenar y evaluar en los mismos datos.
> - Seleccionar variables mirando el test set.
> - Ignorar los supuestos del modelo.
> - Aceptar el primer modelo sin comparar.
> - Usar el modelo sin entender los coeficientes.
> - Implementar sin plan de monitoreo.
> - Olvidar el problema de negocio original.

---
# 6. Resumen y Conceptos Clave

| Concepto | Descripción |
|---|---|
| **Aprendizaje Supervisado** | Aprender de datos históricos con respuestas conocidas |
| **Regresión** | Predecir valores numéricos continuos |
| **Clasificación** | Asignar categorías o clases |
| **Variable Dependiente ($y$)** | Lo que queremos predecir |
| **Variables Independientes ($X$)** | Lo que usamos para predecir |
| **Overfitting** | Modelo muy complejo que memoriza ruido |
| **Underfitting** | Modelo muy simple que no captura patrones |
| **Train/Test Split** | División crucial para evaluar correctamente |
| **Bias–Variance Tradeoff** | Compromiso entre sesgo y varianza |

## Flujo simplificado

$$ \text{Definir} \rightarrow \text{Recolectar} \rightarrow \text{Explorar} \rightarrow \text{Preparar} \rightarrow \text{Entrenar} \rightarrow \text{Evaluar} \rightarrow \text{Iterar} \rightarrow \text{Implementar} $$

## Reglas de oro

1. **Separa train y test desde el inicio** — es la regla más importante.
2. **NUNCA uses test para entrenar** — garantía de evaluación honesta.
3. **Valida supuestos** — los modelos estadísticos tienen requisitos.
4. **Interpreta antes de usar** — los coeficientes deben tener sentido.
5. **Compara múltiples modelos** — el primero rara vez es el mejor.
6. **Piensa en negocio** — métricas técnicas $\neq$ valor de negocio.
7. **Documenta todo** — tu yo futuro te lo agradecerá.

> ### 💡 Cierre
> El aprendizaje supervisado transforma datos históricos en conocimiento predictivo. Dominar estos fundamentos —entender la diferencia entre regresión y clasificación, reconocer overfitting y underfitting, y seguir un flujo de trabajo sistemático— es esencial para cualquier proyecto exitoso de ciencia de datos. La clave no está sólo en aplicar algoritmos, sino en entender profundamente el problema, los datos, y las implicancias de negocio de nuestras decisiones. Con estos fundamentos sólidos, estás preparado para construir modelos estadísticos robustos y útiles para la toma de decisiones informadas.

---
## Referencias bibliográficas

- Hastie, T., Tibshirani, R., & Friedman, J. (2009). *The elements of statistical learning* (2nd ed.). Springer.
- James, G., Witten, D., Hastie, T., & Tibshirani, R. (2021). *An introduction to statistical learning* (2nd ed.). Springer.
- Mitchell, T. (1997). *Machine learning*. McGraw-Hill.
- Géron, A. (2022). *Hands-on machine learning with Scikit-Learn, Keras, and TensorFlow* (3rd ed.). O'Reilly Media.
- Pedregosa, F., et al. (2011). Scikit-learn: Machine learning in Python. *Journal of Machine Learning Research*, 12, 2825–2830.